# Training data from Lichess games: Stockfish score + game result (Kaggle, CPU only)

This notebook **only builds a dataset**, it does not train. Run it with **Save & Run All** on a CPU session (Accelerator: None, Internet on). It does not use GPU hours.

1. Quick check: the parser on an example game, and the first games of the newest month (fails within minutes if something is wrong)
2. Streams the monthly Lichess game databases, newest month first, and keeps only games that
   - were analysed by Lichess with Stockfish (about 6% of all games have a score after every move)
   - are rated and not bullet
   - ended normally (no time forfeit or abandoned game, because then the result says nothing about the position)
   - were played by two players rated at least `min_elo`
3. Picks up to `positions_per_game` **quiet** positions per game, each with the Stockfish score and **the result of the game**
4. Stops at `target_positions` positions or after `max_hours`, and writes `lichess_games.bin`

**Quiet position** means:
- from move 9 on (the first 8 moves are almost the same in every game)
- the side to move is not in check
- the previous move was not a capture (otherwise a recapture is usually pending)
- the move that was played is not a capture or promotion
- the score hardly changes after the move that was played (at most about 100 cp near equality). A big jump means the player missed something, usually a tactic, so the position was not quiet.

**Output format** (the same as the self-play data of `engine-rs.exe datagen`, so the training notebook reads it the same way): 36 bytes per position: the board (32 bytes, two squares per byte), the side to move (1 byte), the Stockfish score in centipawns for white (2 bytes, mate = ±10000) and the result (1 byte: 0 black won, 1 draw, 2 white won).

**Afterwards:** in the training notebook use **Add Input** → *Your Work* → this notebook. The file is then found under `/kaggle/input/`.

In [ ]:
!pip install -q chess zstandard

In [ ]:
import collections
import datetime
import http.client
import json
import math
import multiprocessing
import os
import random
import re
import struct
import time
import urllib.error
import urllib.request

import chess
import numpy as np
import zstandard

## Settings

In [ ]:
notebook_start = time.time()

target_positions = 50_000_000
max_hours = 11
positions_per_game = 10
min_elo = 1800
min_ply = 16
max_label_jump = 0.2
max_months = 24
batch_games = 1000

CP_SCALE = 0.00184104
MATE_CP = 10000

base_url = "https://database.lichess.org/standard/lichess_db_standard_rated_{}.pgn.zst"
out_dir = "/kaggle/working"
out_path = os.path.join(out_dir, "lichess_games.bin")
log_file = os.path.join(out_dir, "lichess_games.log")
num_workers = os.cpu_count()

print("CPU cores:", num_workers)

## Helpers

In [ ]:
def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

## From one game to training positions

A game in the Lichess database looks like this (the score is always for white, `#4` means white mates in 4):

`1. e4 { [%eval 0.18] [%clk 0:05:00] } 1... e5 { [%eval 0.22] [%clk 0:05:00] } 2. Nf3?! { [%eval 0.15] } ... 1-0`

The score after a move is the score of the position after that move. The game is replayed with python-chess, and for every position the filters above decide if it is a candidate. Then up to `positions_per_game` candidates are picked at random.

In [ ]:
RESULTS = {"1-0": 2, "1/2-1/2": 1, "0-1": 0}
TOKEN = re.compile(r"\{([^}]*)\}|(\S+)")
EVAL = re.compile(r"%eval (#?)(-?[\d.]+)")
HEADER = re.compile(rb'\[(\w+) "([^"]*)"\]')
RECORD = struct.Struct("<32sBhB")


def parse_eval(comment):
    match = EVAL.search(comment)
    if match is None:
        return None
    mate, value = match.groups()
    if mate:
        moves = int(value)
        if moves == 0:
            return None
        return MATE_CP if moves > 0 else -MATE_CP
    return max(-MATE_CP, min(MATE_CP, round(float(value) * 100)))


def parse_movetext(text):
    moves, evals = [], []
    for match in TOKEN.finditer(text):
        comment, token = match.groups()
        if comment is not None:
            if moves and evals[-1] is None:
                evals[-1] = parse_eval(comment)
        elif token in RESULTS or token == "*":
            break
        elif not token[0].isdigit() and not token.startswith("$"):
            moves.append(token.rstrip("?!"))
            evals.append(None)
    return moves, evals


def label(cp):
    return math.tanh(cp * CP_SCALE)


def pack(board, cp, result):
    row = [0] * 64
    for square, piece in board.piece_map().items():
        row[square] = piece.piece_type + (0 if piece.color == chess.WHITE else 6)
    packed = bytes(row[i] | (row[i + 1] << 4) for i in range(0, 64, 2))
    return RECORD.pack(packed, int(board.turn == chess.WHITE), cp, result)


def is_candidate(board, ply, move, score, next_score, previous_capture, capture):
    if ply < min_ply or score is None or next_score is None:
        return False
    if previous_capture or capture or move.promotion is not None or board.is_check():
        return False
    return abs(label(score) - label(next_score)) <= max_label_jump


def game_positions(text, result, rng):
    moves, evals = parse_movetext(text)
    board = chess.Board()
    candidates = []
    previous_capture = False
    for ply, san in enumerate(moves):
        try:
            move = board.parse_san(san)
        except ValueError:
            break
        capture = board.is_capture(move)
        score = evals[ply - 1] if ply > 0 else None
        if is_candidate(board, ply, move, score, evals[ply], previous_capture, capture):
            candidates.append(pack(board, score, result))
        previous_capture = capture
        board.push(move)
    return rng.sample(candidates, min(positions_per_game, len(candidates)))


def process_batch(games, seed):
    rng = random.Random(seed)
    records = []
    for movetext, result in games:
        records.extend(game_positions(movetext.decode("utf-8", "replace"), result, rng))
    return b"".join(records), len(records)


def unpack_record(record):
    packed, white, cp, result = RECORD.unpack(record)
    board = chess.Board(None)
    for square in range(64):
        code = (packed[square // 2] >> (4 * (square % 2))) & 15
        if code:
            board.set_piece_at(square, chess.Piece((code - 1) % 6 + 1, code <= 6))
    board.turn = bool(white)
    return board, cp, result

## Streaming a month from Lichess

A month is tens of GB, so it is never stored: it is downloaded, decompressed and split into games on the fly. Only games with `%eval` in them are looked at more closely. The heavy part (replaying the games) runs on all CPU cores at the same time.

If the download of a month breaks off, the positions read so far are kept and the notebook continues with the previous month.

In [ ]:
def available_months(max_months):
    months = []
    today = datetime.date.today()
    year, month = today.year, today.month
    for _ in range(max_months + 3):
        name = f"{year}-{month:02d}"
        request = urllib.request.Request(base_url.format(name), method="HEAD")
        try:
            with urllib.request.urlopen(request, timeout=30):
                months.append(name)
        except urllib.error.URLError:
            pass
        year, month = (year, month - 1) if month > 1 else (year - 1, 12)
    return months[:max_months]


def read_games(reader, chunk_size=16_000_000):
    rest = b"\n\n"
    while True:
        block = reader.read(chunk_size)
        if not block:
            break
        pieces = (rest + block).split(b"\n\n[Event ")
        rest = pieces.pop()
        for piece in pieces:
            if piece:
                yield piece
    if rest.strip():
        yield rest


def select_game(game):
    header, _, movetext = game.partition(b"\n\n")
    tags = dict(HEADER.findall(b"[Event " + header))
    event = tags.get(b"Event", b"")
    if not event.startswith(b"Rated") or b"Bullet" in event:
        return None
    if tags.get(b"Termination") != b"Normal" or b"FEN" in tags:
        return None
    result = RESULTS.get(tags.get(b"Result", b"").decode())
    if result is None:
        return None
    try:
        elos = int(tags[b"WhiteElo"]), int(tags[b"BlackElo"])
    except (KeyError, ValueError):
        return None
    if min(elos) < min_elo:
        return None
    return movetext, result


def process_month(month, pool, out_file, needed, deadline, max_games=None):
    counts = collections.Counter()
    pending = collections.deque()
    batch = []
    try:
        with urllib.request.urlopen(base_url.format(month), timeout=60) as response:
            reader = zstandard.ZstdDecompressor().stream_reader(response, read_across_frames=True)
            for game in read_games(reader):
                counts["games"] += 1
                if b"%eval" in game:
                    counts["analysed"] += 1
                    selected = select_game(game)
                    if selected is not None:
                        batch.append(selected)
                if len(batch) == batch_games:
                    pending.append(pool.apply_async(process_batch, (batch, f"{month}-{counts['games']}")))
                    counts["kept"] += len(batch)
                    batch = []
                while len(pending) > 2 * num_workers or (pending and pending[0].ready()):
                    data, num = pending.popleft().get()
                    out_file.write(data)
                    counts["positions"] += num
                if counts["positions"] >= needed or time.time() > deadline or counts["games"] == max_games:
                    break
    except (OSError, http.client.HTTPException, zstandard.ZstdError) as error:
        log(f"{month}: download stopped early ({type(error).__name__}: {error}), keeping what was read")
    if batch:
        pending.append(pool.apply_async(process_batch, (batch, f"{month}-{counts['games']}")))
        counts["kept"] += len(batch)
    while pending:
        data, num = pending.popleft().get()
        out_file.write(data)
        counts["positions"] += num
    return counts


def month_summary(month, counts):
    return f"{month}: {counts['games']:,} games, {counts['analysed']:,} with Stockfish scores, {counts['kept']:,} kept, {counts['positions']:,} positions"

## Quick check

In [ ]:
example = (
    "1. e4 { [%eval 0.18] [%clk 0:05:00] } 1... e5 { [%eval 0.22] [%clk 0:05:00] } 2. Nf3 { [%eval 0.15] } 2... Nc6 { [%eval 0.2] } "
    "3. Bb5 { [%eval 0.25] } 3... a6 { [%eval 0.3] } 4. Ba4 { [%eval 0.28] } 4... Nf6 { [%eval 0.3] } 5. O-O { [%eval 0.25] } "
    "5... Be7 { [%eval 0.3] } 6. Re1 { [%eval 0.3] } 6... b5 { [%eval 0.35] } 7. Bb3 { [%eval 0.3] } 7... d6 { [%eval 0.35] } "
    "8. c3 { [%eval 0.3] } 8... O-O { [%eval 0.3] } 9. h3 { [%eval 0.32] } 9... Nb8?! { [%eval 0.6] } 10. d4 { [%eval 0.55] } "
    "10... Nbd7 { [%eval 0.5] } 11. Nbd2 { [%eval 0.45] } 11... Bb7 { [%eval 0.5] } 12. Bc2 { [%eval 2.5] } 12... Re8 { [%eval 0.5] } "
    "13. Nf1 { [%eval 0.45] } 13... Bf8 { [%eval #4] } 1-0"
)
moves, evals = parse_movetext(example)
assert len(moves) == 26 and moves[17] == "Nb8", moves
assert evals[0] == 18 and evals[22] == 250 and evals[25] == MATE_CP, evals

records = game_positions(example, 2, random.Random(0))
board = chess.Board()
expected = {}
for ply, san in enumerate(moves):
    if ply in (16, 17, 18, 19, 20, 21, 24):
        expected[board.board_fen() + (" w" if board.turn else " b")] = evals[ply - 1]
    board.push_san(san)
found = {}
for record in records:
    position, cp, result = unpack_record(record)
    assert result == 2
    found[position.board_fen() + (" w" if position.turn else " b")] = cp
assert found == expected, (found, expected)
log(f"Example game: {len(moves)} moves, {len(records)} quiet positions, as expected")

months = available_months(max_months)
assert months, "no Lichess months found: is Internet on in the notebook settings?"
log(f"Available months, newest first: {months}")

check_path = os.path.join(out_dir, "check.bin")
with open(check_path, "wb") as check_file, multiprocessing.get_context("fork").Pool(num_workers) as pool:
    start = time.time()
    counts = process_month(months[0], pool, check_file, needed=10**9, deadline=time.time() + 600, max_games=200_000)
log(month_summary(months[0], counts) + f" in {time.time() - start:.0f} s")
assert counts["analysed"] > 0 and counts["positions"] > 0, "no positions found in the first games"

with open(check_path, "rb") as f:
    data = f.read()
os.remove(check_path)
assert len(data) == counts["positions"] * RECORD.size
for i in range(0, min(len(data), 2000 * RECORD.size), RECORD.size):
    position, cp, result = unpack_record(data[i:i + RECORD.size])
    assert position.is_valid() and not position.is_check(), position.fen()
    assert len(position.pieces(chess.KING, chess.WHITE)) == 1 and len(position.pieces(chess.KING, chess.BLACK)) == 1
    assert abs(cp) <= MATE_CP and result in (0, 1, 2)
games_per_hour = counts["games"] / (time.time() - start) * 3600
log(f"Quick check passed. Speed: about {games_per_hour / 1e6:.0f} million games per hour, {counts['positions'] / max(1, counts['games']) * 1000:.0f} positions per 1000 games")

## Building the dataset

Month after month, newest first, until `target_positions` positions are found or `max_hours` is reached. The file is written while the notebook runs, so even a stopped run leaves a usable file.

In [ ]:
deadline = notebook_start + max_hours * 3600
total = collections.Counter()
per_month = {}

with open(out_path, "wb") as out_file, multiprocessing.get_context("fork").Pool(num_workers) as pool:
    for month in months:
        if total["positions"] >= target_positions or time.time() > deadline:
            break
        log(f"{month}: streaming...")
        counts = process_month(month, pool, out_file, target_positions - total["positions"], deadline)
        total.update(counts)
        per_month[month] = dict(counts)
        log(month_summary(month, counts) + f" (total {total['positions']:,})")

log(f"Done: {total['positions']:,} positions from {total['kept']:,} games")

## Statistics

In [ ]:
GAMES_RECORD = np.dtype([("board", "u1", 32), ("white", "u1"), ("score", "<i2"), ("result", "u1")])
records = np.fromfile(out_path, dtype=GAMES_RECORD)
scores = records["score"].astype(np.int32)
mates = np.abs(scores) == MATE_CP
result_share = np.bincount(records["result"], minlength=3) / len(records)

log(f"File: {os.path.getsize(out_path) / 1e9:.2f} GB, {len(records):,} positions")
log(f"Results (black won, draw, white won): {np.round(result_share, 3)}")
log(f"White to move: {records['white'].mean():.3f}")
log(f"Mate scores: {mates.mean():.3%}, median |score| without mates: {np.median(np.abs(scores[~mates])):.0f} cp")

summary = {
    "params": {
        "target_positions": target_positions,
        "positions_per_game": positions_per_game,
        "min_elo": min_elo,
        "min_ply": min_ply,
        "max_label_jump": max_label_jump,
    },
    "total": dict(total),
    "per_month": per_month,
    "result_share": result_share.tolist(),
}
with open(os.path.join(out_dir, "lichess_games.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
log(f"Saved lichess_games.json. Files: {sorted(os.listdir(out_dir))}")